In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.optimize import minimize
import matplotlib.gridspec as gridspec
from scipy import stats

In [2]:
def research(x):
    """
        Grows logarithmically from 0 to 200k as x goes from 0 to 100
        x: int, the research percentage (0 to 100)
    """
    return 200_000 * np.log(1 + x) / np.log(1 + 100)

def scale(x):
    """
        Grows linearly from 0 to 7 as x goes from 0 to 100

        x: int, the scale percentage (0 to 100)
    """
    return 7 * x / 100

# Neglecting speed distribution

In [ ]:
def speed_direct_mapping(x):
    """
    Linear proxy (rank-based to be implemented later).
    """
    return 0.1 + 0.8 * x / 100

def speed_real(x):
    """
    Rank-based across all players:
    - Highest speed investment receives a `0.9` multiplier.
    - Lowest receives `0.1`.
    - Everyone in between is scaled linearly by rank, equal investments share the same rank.
    """
    pass

BUDGET = 50_000

def PnL(r_pct, sc_pct, sp_pct):
    """
    r_pct, sc_pct, sp_pct: floats in [0, 100], must sum to <= 100.
    Budget used = 50_000 * (total allocation as a fraction).
    """
    total_alloc = r_pct + sc_pct + sp_pct          # in [0, 300], constrained to ≤ 100
    budget_used = BUDGET * (total_alloc / 100)
    gross = research(r_pct) * scale(sc_pct) * speed_direct_mapping(sp_pct)
    return gross - budget_used
    

In [3]:
r_pct, sc_pct, sp_pct = 40, 10, 40

pnl = PnL(r_pct, sc_pct, sp_pct)
print(round(pnl, 2))

2313.62


In [4]:


def neg_pnl(x):
    """Objective for scipy (minimizes, so we negate)."""
    return -PnL(*x)

result = minimize(
    neg_pnl,
    x0=[40, 20, 40],                               # starting guess
    method="SLSQP",
    bounds=[(0, 100), (0, 100), (0, 100)],          # each pillar in [0, 100]
    constraints=[
        {"type": "ineq", "fun": lambda x: 100 - sum(x)}   # sum ≤ 100 (not =)
    ],
    options={"ftol": 1e-12, "maxiter": 10_000}
)

r_opt, sc_opt, sp_opt = result.x
total_opt = sum(result.x)
budget_used  = BUDGET * (total_opt / 100)
budget_saved = BUDGET - budget_used

print(f"Research : {r_opt:.2f}%")
print(f"Scale    : {sc_opt:.2f}%")
print(f"Speed    : {sp_opt:.2f}%")
print(f"Total    : {total_opt:.2f}%  (budget saved: {budget_saved:,.0f} XIRECs)")
print(f"Net PnL  : {-result.fun:,.0f}")

Research : 16.02%
Scale    : 48.24%
Speed    : 35.74%
Total    : 100.00%  (budget saved: -0 XIRECs)
Net PnL  : 110,070


In [15]:
np.random.seed(42)
N = 500
results = []

for _ in range(N):
    a, b, c = np.random.dirichlet([1,1,1]) * np.random.uniform(0, 100)
    res = minimize(
        neg_pnl, [a, b, c],
        method="SLSQP",
        bounds=[(0,100),(0,100),(0,100)],
        constraints=[{"type":"ineq","fun": lambda x: 100 - sum(x)}],
        options={"ftol":1e-12,"maxiter":10_000}
    )
    if res.success or res.fun < 0:
        r, sc, sp = res.x
        results.append({"r": r, "sc": sc, "sp": sp, "pnl": -res.fun, "total": r+sc+sp})

df = pd.DataFrame(results)
is_zero = df["total"] < 1
is_opt  = ~is_zero

# --- 4-panel stability figure ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Research % across runs",
        "Scale % across runs",
        "Speed % across runs",
        "Net PnL distribution",
    ]
)

idx = list(range(len(df)))

for col_name, row, col, title in [("r",1,1,"Research %"), ("sc",1,2,"Scale %"), ("sp",2,1,"Speed %")]:
    fig.add_trace(go.Scatter(
        x=[i for i in idx if is_opt.iloc[i]],
        y=df[col_name][is_opt].values,
        mode="markers", marker=dict(color="#1D9E75", size=4, opacity=0.5),
        name="converged", legendgroup="converged", showlegend=(col_name=="r")
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=[i for i in idx if is_zero.iloc[i]],
        y=df[col_name][is_zero].values,
        mode="markers", marker=dict(color="#E24B4A", size=4, opacity=0.7),
        name="trivial (zero)", legendgroup="zero", showlegend=(col_name=="r")
    ), row=row, col=col)

# PnL histogram
fig.add_trace(go.Histogram(
    x=df["pnl"][is_opt],
    nbinsx=40, marker_color="#1D9E75", opacity=0.8,
    name="PnL (converged)", showlegend=False
), row=2, col=2)
fig.add_trace(go.Histogram(
    x=df["pnl"][is_zero],
    nbinsx=5, marker_color="#E24B4A", opacity=0.8,
    name="PnL (trivial)", showlegend=False
), row=2, col=2)

fig.update_layout(
    title="Optimizer stability — 500 random restarts",
    height=700, barmode="overlay",
    legend=dict(orientation="h", y=-0.08)
)
fig.show()


In [5]:
r_opt, sc_opt, sp_opt = 16.02, 48.24, 35.74
grid = np.linspace(0, 100, 200)

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type":"surface"}, {"type":"surface"}, {"type":"surface"}]],
    subplot_titles=[
        f"PnL(R, SC)  — speed fixed at {sp_opt}%",
        f"PnL(R, SP)  — scale fixed at {sc_opt}%",
        f"PnL(SC, SP) — research fixed at {r_opt}%",
    ]
)

# panel 1: sweep R and SC, fix speed
R, SC = np.meshgrid(grid, grid)
mask = R + SC <= 100
Z1 = np.where(mask, PnL(R, SC, sp_opt), np.nan)

fig.add_trace(go.Surface(x=grid, y=grid, z=Z1,
    colorscale="Viridis", showscale=False, name="R vs SC"), row=1, col=1)

# panel 2: sweep R and SP, fix scale
R, SP = np.meshgrid(grid, grid)
mask = R + SP <= 100
Z2 = np.where(mask, PnL(R, sc_opt, SP), np.nan)

fig.add_trace(go.Surface(x=grid, y=grid, z=Z2,
    colorscale="RdYlBu", showscale=False, name="R vs SP"), row=1, col=2)

# panel 3: sweep SC and SP, fix research
SC, SP = np.meshgrid(grid, grid)
mask = SC + SP <= 100
Z3 = np.where(mask, PnL(r_opt, SC, SP), np.nan)

fig.add_trace(go.Surface(x=grid, y=grid, z=Z3,
    colorscale="Plasma", showscale=False, name="SC vs SP"), row=1, col=3)

fig.update_xaxes(title_text="Research %", row=1, col=1)
fig.update_yaxes(title_text="Scale %", row=1, col=1)
fig.update_scenes(xaxis_title="Research %", yaxis_title="Scale %", zaxis_title="PnL", row=1, col=1)

fig.update_scenes(xaxis_title="Research %", yaxis_title="Speed %", zaxis_title="PnL", row=1, col=2)
fig.update_scenes(xaxis_title="Scale %", yaxis_title="Speed %", zaxis_title="PnL", row=1, col=3)

fig.update_layout(title="PnL surface — one parameter fixed at optimum", height=500)
fig.show()

# What speed allocation is worth paying for, given what I expect others to pay?

### Step 1 — Establish your baseline without speed competition


#### Given I've committed b% to speed, what's my best R and SC?

In [ ]:
# fix speed multiplier at a given value, optimize only R and SC
def PnL_fixed_speed(r_pct, sc_pct, speed_multiplier):
    budget_used = BUDGET * ((r_pct + sc_pct) / 100)
    return research(r_pct) * scale(sc_pct) * speed_multiplier - budget_used

# step 1: store optimal R, SC for each (sp_pct, speed_mult) combination
records = []

for sp_pct in np.linspace(0, 50, 11): # sweep how much you commit to speed
    for speed_mult in np.linspace(0.1, 0.9, 18): # sweep what rank you might get for it
        res = minimize(
            lambda x, sm=speed_mult, sp=sp_pct: -PnL_fixed_speed(x[0], x[1], sm),
            x0=[16, 48],
            method="SLSQP",
            bounds=[(0, 100-sp_pct), (0, 100-sp_pct)],
            constraints=[{"type":"ineq","fun": lambda x, sp=sp_pct: (100 - sp) - sum(x)}],
            options={"ftol":1e-12}
        )
        r, sc = res.x
        pnl = -res.fun - BUDGET * (sp_pct / 100)  # deduct speed cost from PnL
        # print(f"sp_pct={sp_pct:.0f}%  mult={speed_mult:.1f} → "
        #       f"R={r:.1f}%  SC={sc:.1f}%  PnL={pnl:,.0f} - Ratio of r/(r+sc): {r/(r+sc):.2f}")
        records.append({
            "sp_pct":     sp_pct,
            "speed_mult": speed_mult,
            "r":          r,
            "sc":         sc,
            "ratio":      r / (r + sc) if (r + sc) > 0 else np.nan,
            "pnl":        pnl,
        })

df_step1 = pd.DataFrame(records)
df_step1

sp_pct=0%  mult=0.1 → R=23.1%  SC=76.9%  PnL=24,234 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.1 → R=23.1%  SC=76.9%  PnL=59,167 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.2 → R=23.2%  SC=76.9%  PnL=94,177 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.2 → R=23.1%  SC=76.9%  PnL=129,034 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.3 → R=23.1%  SC=76.9%  PnL=163,968 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.3 → R=23.2%  SC=76.9%  PnL=199,093 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.4 → R=23.1%  SC=76.9%  PnL=233,835 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.4 → R=23.1%  SC=76.9%  PnL=268,791 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.5 → R=23.1%  SC=76.9%  PnL=303,702 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.5 → R=23.1%  SC=76.9%  PnL=338,635 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.6 → R=23.1%  SC=76.9%  PnL=373,569 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.6 → R=23.1%  SC=76.9%  PnL=408,513 - Ratio of r/(r+sc): 0.23
sp_pct=0%  mult=0.7 → R=23.1%  SC=76.9%  PnL=443,447 - 

In [29]:
import numpy as np
import plotly.graph_objects as go

sp_pcts     = np.linspace(0, 50, 50)
speed_mults = np.linspace(0.1, 0.9, 50)

ratio_grid = np.zeros((len(speed_mults), len(sp_pcts)))

for j, sp_pct in enumerate(sp_pcts):
    for i, speed_mult in enumerate(speed_mults):
        res = minimize(
            lambda x: -PnL_fixed_speed(x[0], x[1], speed_mult),
            x0=[16, 48],
            method="SLSQP",
            bounds=[(0, 100-sp_pct), (0, 100-sp_pct)],
            constraints=[{"type":"ineq","fun": lambda x: (100 - sp_pct) - sum(x)}],
            options={"ftol":1e-12}
        )
        r, sc = res.x
        ratio_grid[i, j] = r / (r + sc) if (r + sc) > 0 else np.nan

fig = go.Figure()

fig.add_trace(go.Surface(
    z=ratio_grid,
    x=sp_pcts,
    y=speed_mults,
    colorscale="Teal",
    showscale=True,
    colorbar=dict(title="R/(R+SC)", thickness=20, x=1.0),
    name="R/(R+SC)",
    showlegend=True,
))

fig.update_layout(
    title="R / (R+SC) — share of non-speed budget going to Research",
    height=600,
    margin=dict(l=10, r=10, t=60, b=10),
    legend=dict(x=0.75, y=0.95),
    scene=dict(
        xaxis_title="Speed spend %",
        yaxis_title="Speed multiplier",
        zaxis_title="R / (R+SC)",
    ),
)

fig.show()


In [ ]:

# step 2: marginal value of speed multiplier, now using the correct R and SC for each point
# for each sp_pct, compute dPnL/d(speed_mult) using the stored optimal r, sc
records_mv = []

for sp_pct, group in df_step1.groupby("sp_pct"):
    group = group.sort_values("speed_mult").reset_index(drop=True)
    for idx in range(1, len(group)):
        r_mid  = (group.loc[idx, "r"]  + group.loc[idx-1, "r"])  / 2
        sc_mid = (group.loc[idx, "sc"] + group.loc[idx-1, "sc"]) / 2
        dm     = group.loc[idx, "speed_mult"] - group.loc[idx-1, "speed_mult"]
        dpnl   = group.loc[idx, "pnl"]        - group.loc[idx-1, "pnl"]
        records_mv.append({
            "sp_pct":        sp_pct,
            "speed_mult_mid": (group.loc[idx, "speed_mult"] + group.loc[idx-1, "speed_mult"]) / 2,
            "marginal_value": dpnl / dm,
            "r":             r_mid,
            "sc":            sc_mid,
        })

df_mv = pd.DataFrame(records_mv)

# plot: one line per sp_pct
fig = go.Figure()
for sp_pct, group in df_mv.groupby("sp_pct"):
    fig.add_trace(go.Scatter(
        x=group["speed_mult_mid"],
        y=group["marginal_value"],
        mode="lines",
        name=f"sp={sp_pct:.0f}%",
    ))

fig.update_layout(
    title="Marginal value of speed multiplier (at optimal R/SC for each sp_pct)",
    xaxis_title="Speed multiplier",
    yaxis_title="PnL gained per unit of multiplier",
    height=500
)
fig.show()

The marginal value of the speed multiplier is:

$$
\frac{d\,\mathrm{PnL}}{d\,\mathrm{speed\_mult}} = \mathrm{research}(r) \cdot \mathrm{scale}(sc)
$$

In [39]:
fig = go.Figure()

for sp_pct, group in df_step1.groupby("sp_pct"):
    group = group.sort_values("speed_mult")
    fig.add_trace(go.Scatter(
        x=group["speed_mult"],
        y=group["pnl"],
        mode="lines",
        name=f"sp={sp_pct:.0f}%",
    ))

fig.update_layout(
    title="Absolute PnL vs speed multiplier, for each speed spend level",
    xaxis_title="Speed multiplier received",
    yaxis_title="Net PnL",
    height=500
)
fig.show()

In [7]:
def PnL_real(r_pct, sc_pct, sp_pct, others_sp):
    total_alloc = r_pct + sc_pct + sp_pct
    budget_used = BUDGET * (total_alloc / 100)
    gross = research(r_pct) * scale(sc_pct) * speed_real(sp_pct, others_sp)
    return gross - budget_used

def best_RSC_given_speed_budget(b, others_sp):
    """Given I've committed b% to speed, what's my best R and SC?"""
    def obj(x):
        r, sc = x
        return -PnL_real(r, sc, b, others_sp)
    
    res = minimize(
        obj,
        x0=[16, 48],
        method="SLSQP",
        bounds=[(0, 100-b), (0, 100-b)],
        constraints=[{"type":"ineq", "fun": lambda x: (100 - b) - sum(x)}],
        options={"ftol":1e-12}
    )
    return -res.fun, res.x  # best PnL and (r, sc) for this speed budget

In [8]:
best_RSC_given_speed_budget(40, [10, 20, 30])

(np.float64(290700.18117408094), array([15.1359077 , 44.87267887]))

In [42]:
summary = []
baseline_pnl = df_step1[df_step1["sp_pct"] == 0]["pnl"].max()

for sp_pct, group in df_step1.groupby("sp_pct"):
    summary.append({
        "sp_pct":    sp_pct,
        "pnl_worst": group["pnl"].min(),   # you spent sp_pct% and got rank=last (mult=0.1)
        "pnl_best":  group["pnl"].max(),   # you spent sp_pct% and got rank=first (mult=0.9)
        "pnl_mid":   group["pnl"].mean(),  # average across all multipliers
    })

df_summary = pd.DataFrame(summary)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_summary["sp_pct"], y=df_summary["pnl_best"],
    name="best case (rank 1, mult=0.9)", line=dict(color="#1D9E75")))
fig.add_trace(go.Scatter(x=df_summary["sp_pct"], y=df_summary["pnl_mid"],
    name="average rank", line=dict(color="#378ADD")))
fig.add_trace(go.Scatter(x=df_summary["sp_pct"], y=df_summary["pnl_worst"],
    name="worst case (rank last, mult=0.1)", line=dict(color="#E24B4A")))
fig.add_hline(y=baseline_pnl, line_dash="dot", line_color="#888",
    annotation_text=f"baseline: sp=0% (no speed spend): {baseline_pnl:,.0f}")

fig.update_layout(
    title="PnL range vs speed spend — best/avg/worst rank outcome",
    xaxis_title="Speed spend %",
    yaxis_title="Net PnL",
    height=500
)
fig.show()

# Finding the Nash equilibrium

In [12]:
def mixture_cdf_with_rational_sp(speed_val, rational_sp, 
                                  proportions, n_samples=20_000):
    """
    Same as mixture_cdf but Type C plays rational_sp (the candidate Nash speed)
    instead of the hardcoded 33.33. This breaks the circular dependency.
    """
    pi_a, pi_b, pi_c = proportions
    n_a = int(pi_a * n_samples)
    n_b = int(pi_b * n_samples)
    n_c = n_samples - n_a - n_b

    population = np.concatenate([
        sample_type_A(n_a),
        sample_type_B(n_b),
        np.clip(np.random.normal(rational_sp, 2.0, n_c), 0, 100)  # Type C at candidate Nash
    ])
    return np.mean(population <= speed_val)

def find_nash_heterogeneous(dirichlet_alpha, n_trials=200, tol=0.5, max_iter=30):
    """
    Fixed point iteration over rational SP, with heterogeneous population.
    
    At each candidate SP_candidate:
    - Types A and B play their fixed distributions
    - Type C plays SP_candidate
    - You optimize (R, SC, SP) against this population
    - Your optimal SP becomes the next candidate
    - Repeat until SP_candidate stops moving
    """
    sp_candidate = 35.74  # start from naive optimum
    history = []

    for i in range(max_iter):
        def obj(x):
            r, sc, sp = x
            pnls = []
            for _ in range(n_trials):
                proportions = np.random.dirichlet(dirichlet_alpha)
                percentile  = mixture_cdf_with_rational_sp(
                                sp, sp_candidate, proportions)
                multiplier  = 0.1 + 0.8 * percentile
                budget_used = BUDGET * (r + sc + sp) / 100
                pnls.append(research(r) * scale(sc) * multiplier - budget_used)
            return -np.mean(pnls)

        res = minimize(
            obj, x0=[16, 48, sp_candidate],
            method="SLSQP",
            bounds=[(0,100),(0,100),(0,100)],
            constraints=[{"type":"ineq","fun": lambda x: 100 - sum(x)}],
            options={"ftol":1e-4, "maxiter":200}
        )
        r_new, sc_new, sp_new = res.x
        delta = abs(sp_new - sp_candidate)

        print(f"iter {i+1:3d} | R={r_new:.2f}%  SC={sc_new:.2f}%  "
              f"SP={sp_new:.2f}% | delta={delta:.4f}")
        history.append({"iter": i+1, "r": r_new, "sc": sc_new, 
                        "sp": sp_new, "delta": delta})

        if delta < tol:
            print(f"\nConverged: Nash SP* = {sp_new:.2f}%")
            break

        sp_candidate = sp_new

    return pd.DataFrame(history), (r_new, sc_new, sp_new)

In [13]:
scenarios = {
    "uniform belief":        [1,  1,  1],
    "naive dominated":       [20, 5,  1],
    "nash dominated":        [20, 5, 75],
    "nash dominated_2":      [10, 10, 80],
    "nash dominated_3":      [5, 5, 90],
}

nash_results = {}
for name, alpha in scenarios.items():
    print(f"\n{'='*60}")
    print(f"Scenario: {name}  alpha={alpha}")
    print(f"{'='*60}")

    history_df, (r_star, sc_star, sp_star) = find_nash_heterogeneous(
        dirichlet_alpha=alpha,
        n_trials=200,    # increase to 1000+ for more stable gradients, slower
        tol=0.5,         # tighten to 0.1 once you confirm it converges
        max_iter=30
    )

    nash_results[name] = {
        "r": r_star, "sc": sc_star, "sp": sp_star,
        "history": history_df
    }

    print(f"\nNash equilibrium for '{name}':")
    print(f"  R*  = {r_star:.2f}%")
    print(f"  SC* = {sc_star:.2f}%")
    print(f"  SP* = {sp_star:.2f}%")


Scenario: uniform belief  alpha=[1, 1, 1]
iter   1 | R=16.00%  SC=48.00%  SP=35.74% | delta=0.0000

Converged: Nash SP* = 35.74%

Nash equilibrium for 'uniform belief':
  R*  = 16.00%
  SC* = 48.00%
  SP* = 35.74%

Scenario: naive dominated  alpha=[20, 5, 1]
iter   1 | R=16.00%  SC=48.00%  SP=35.74% | delta=0.0000

Converged: Nash SP* = 35.74%

Nash equilibrium for 'naive dominated':
  R*  = 16.00%
  SC* = 48.00%
  SP* = 35.74%

Scenario: nash dominated  alpha=[20, 5, 75]
iter   1 | R=16.00%  SC=48.00%  SP=35.74% | delta=0.0000

Converged: Nash SP* = 35.74%

Nash equilibrium for 'nash dominated':
  R*  = 16.00%
  SC* = 48.00%
  SP* = 35.74%

Scenario: nash dominated_2  alpha=[10, 10, 80]
iter   1 | R=16.08%  SC=48.05%  SP=35.80% | delta=0.0628

Converged: Nash SP* = 35.80%

Nash equilibrium for 'nash dominated_2':
  R*  = 16.08%
  SC* = 48.05%
  SP* = 35.80%

Scenario: nash dominated_3  alpha=[5, 5, 90]
iter   1 | R=16.00%  SC=48.00%  SP=35.74% | delta=0.0000

Converged: Nash SP* = 35

### Option A — CMA-ES (gradient-free, handles noise well)


In [16]:
import cma

def find_nash_cma(dirichlet_alpha, n_trials=500, tol=0.5, max_iter=30):
    sp_candidate = 35.74
    history = []

    for i in range(max_iter):
        def obj(x):
            r, sc, sp = x
            # enforce constraints via penalty
            if r < 0 or sc < 0 or sp < 0 or r+sc+sp > 100:
                return 1e9
            pnls = []
            for _ in range(n_trials):
                proportions = np.random.dirichlet(dirichlet_alpha)
                percentile  = mixture_cdf_with_rational_sp(sp, sp_candidate, proportions)
                multiplier  = 0.1 + 0.8 * percentile
                budget_used = BUDGET * (r + sc + sp) / 100
                pnls.append(research(r) * scale(sc) * multiplier - budget_used)
            return -np.mean(pnls)

        es = cma.CMAEvolutionStrategy(
            x0=[16, 48, sp_candidate],
            sigma0=5,                        # initial search radius
            inopts={
                "bounds": [[0,0,0],[100,100,100]],
                "tolx": 0.1,
                "tolfun": 10,
                "maxiter": 100,
                "verbose": -9             # silent
            }
        )
        es.optimize(obj)
        r_new, sc_new, sp_new = es.result.xbest
        # enforce sum <= 100 manually if needed
        if r_new + sc_new + sp_new > 100:
            scale_factor = 100 / (r_new + sc_new + sp_new)
            r_new *= scale_factor
            sc_new *= scale_factor
            sp_new *= scale_factor

        delta = abs(sp_new - sp_candidate)
        print(f"iter {i+1:3d} | R={r_new:.2f}%  SC={sc_new:.2f}%  "
              f"SP={sp_new:.2f}% | delta={delta:.4f}")
        history.append({"iter": i+1, "r": r_new, "sc": sc_new,
                        "sp": sp_new, "delta": delta})

        if delta < tol:
            print(f"\nConverged: Nash SP* = {sp_new:.2f}%")
            break
        sp_candidate = sp_new

    return pd.DataFrame(history), (r_new, sc_new, sp_new)

In [ ]:
nash_results = {}
for name, alpha in scenarios.items():
    print(f"\n{'='*60}")
    print(f"Scenario: {name}  alpha={alpha}")
    print(f"{'='*60}")

    history_df, (r_star, sc_star, sp_star) = find_nash_cma(
        dirichlet_alpha=alpha,
        n_trials=2000,    # increase to 1000+ for more stable gradients, slower
        tol=0.5,         # tighten to 0.1 once you confirm it converges
        max_iter=30
    )

    nash_results[name] = {
        "r": r_star, "sc": sc_star, "sp": sp_star,
        "history": history_df
    }

    print(f"\nNash equilibrium for '{name}':")
    print(f"  R*  = {r_star:.2f}%")
    print(f"  SC* = {sc_star:.2f}%")
    print(f"  SP* = {sp_star:.2f}%")


Scenario: uniform belief  alpha=[1, 1, 1]
iter   1 | R=14.83%  SC=44.93%  SP=40.15% | delta=4.4132
iter   2 | R=14.17%  SC=42.03%  SP=43.64% | delta=3.4895
iter   3 | R=12.57%  SC=40.33%  SP=46.95% | delta=3.3114
iter   4 | R=9.31%  SC=42.99%  SP=47.70% | delta=0.7432
iter   5 | R=13.07%  SC=36.57%  SP=50.32% | delta=2.6191
iter   6 | R=11.94%  SC=34.39%  SP=53.57% | delta=3.2539
iter   7 | R=15.74%  SC=37.67%  SP=46.60% | delta=6.9750
iter   8 | R=15.71%  SC=34.85%  SP=49.39% | delta=2.7969
iter   9 | R=9.45%  SC=45.71%  SP=44.84% | delta=4.5536
iter  10 | R=16.80%  SC=40.90%  SP=42.05% | delta=2.7919
iter  11 | R=13.24%  SC=40.35%  SP=46.06% | delta=4.0148
iter  12 | R=12.94%  SC=43.18%  SP=43.88% | delta=2.1810
iter  13 | R=17.96%  SC=41.24%  SP=40.79% | delta=3.0884
iter  14 | R=14.38%  SC=44.08%  SP=41.55% | delta=0.7530
iter  15 | R=13.69%  SC=40.94%  SP=45.33% | delta=3.7839
iter  16 | R=12.82%  SC=38.69%  SP=48.41% | delta=3.0773
iter  17 | R=13.13%  SC=35.22%  SP=51.42% | del

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import cma

BUDGET  = 50_000
N_TEAMS = 20_000

# ── primitive functions ────────────────────────────────────────────────────────

def research(x):
    return 200_000 * np.log(1 + x) / np.log(1 + 100)

def scale(x):
    return 7 * x / 100

def speed_multiplier_from_rank(rank, n=N_TEAMS):
    """
    m(rank) = 0.9 - 0.8 × (rank-1) / (N-1)
    rank 1    → 0.9
    rank N    → 0.1
    rank 2    → 0.9 - 0.8/(N-1)  ← almost 0.9 for large N
    """
    return 0.9 - 0.8 * (rank - 1) / (n - 1)

# ── behavioral type samplers ───────────────────────────────────────────────────

def sample_type_A(n):
    """Naive: ran direct-mapping optimizer, noise around 35.74%"""
    return np.clip(np.random.normal(loc=35.74, scale=4.0, size=n), 0, 100)

def sample_type_B(n):
    """Uninformed rational: uniform — maximum entropy prior"""
    return np.random.uniform(0, 100, size=n)

def sample_type_C(n, nash_sp):
    """
    Coordinated rational: play near nash_sp.
    nash_sp passed explicitly — no hardcoding.
    """
    return np.clip(np.random.normal(loc=nash_sp, scale=2.0, size=n), 0, 100)

# ── population CDF ─────────────────────────────────────────────────────────────

def mixture_cdf_with_rational_sp(speed_val, rational_sp,
                                  proportions, n_samples=N_TEAMS):
    """
    Generate a population of N_TEAMS players where:
      Type A ~ Normal(35.74, 4)      — naive players
      Type B ~ Uniform(0, 100)       — uninformed rational
      Type C ~ Normal(rational_sp,2) — coordinated rational (candidate Nash)

    Returns: fraction of population with speed <= speed_val (your percentile)
    """
    pi_a, pi_b, pi_c = proportions
    n_a = int(pi_a * n_samples)
    n_b = int(pi_b * n_samples)
    n_c = n_samples - n_a - n_b

    population = np.concatenate([
        sample_type_A(n_a),
        sample_type_B(n_b),
        sample_type_C(n_c, rational_sp),
    ])
    return np.mean(population <= speed_val)

# ── R/SC optimizer for fixed SP and multiplier ────────────────────────────────

def optimize_r_sc(sp_pct, multiplier):
    """
    Given fixed SP and fixed speed multiplier,
    find optimal R and SC analytically (no noise — SLSQP works fine here).
    Returns (max_pnl, [r_opt, sc_opt])
    """
    def obj(x):
        r, sc = x
        budget_used = BUDGET * (r + sc + sp_pct) / 100
        return -(research(r) * scale(sc) * multiplier - budget_used)

    res = minimize(
        obj,
        x0=[16, 48],
        method="SLSQP",
        bounds=[(0, 100 - sp_pct), (0, 100 - sp_pct)],
        constraints=[{"type": "ineq", "fun": lambda x: (100 - sp_pct) - sum(x)}],
        options={"ftol": 1e-12}
    )
    return -res.fun, res.x


# ═══════════════════════════════════════════════════════════════════════════════
# PART 1 — Symmetric Nash check
# ═══════════════════════════════════════════════════════════════════════════════

def deviation_gain(z_star, n=N_TEAMS):
    """
    With N teams all at z_star (tied rank 1, m=0.9):

    Upward deviation to z_star+1:
      deviator → rank 1 alone
      others   → rank 2  → m = 0.9 - 0.8/(N-1)   ← almost identical for large N
      cost     → 500 XIRECs extra budget

    Downward deviation to 0:
      deviator → rank N  → m = 0.1
      full budget freed for R and SC

    Returns dict with both deviation analyses.
    """
    # baseline: conform at z_star, everyone ties → m=0.9
    pnl_conform, (r_c, sc_c) = optimize_r_sc(z_star, multiplier=0.9)

    # upward deviation: z_star+1, still rank 1, but costs 500 more XIRECs
    if z_star < 100:
        pnl_up, (r_up, sc_up) = optimize_r_sc(z_star + 1, multiplier=0.9)
        gain_up = pnl_up - pnl_conform
    else:
        gain_up = -np.inf

    # downward deviation: SP=0, rank N → m=0.1, but free budget for R/SC
    pnl_down, (r_down, sc_down) = optimize_r_sc(0, multiplier=0.1)
    gain_down = pnl_down - pnl_conform

    # multiplier at rank 2 (what others get when you deviate up)
    m_rank2 = speed_multiplier_from_rank(2, n)

    return {
        "z_star":       z_star,
        "pnl_conform":  pnl_conform,
        "r_conform":    r_c,
        "sc_conform":   sc_c,
        "gain_up":      gain_up,        # should be ≤ 0 for Nash
        "gain_down":    gain_down,      # should be ≤ 0 for Nash
        "m_rank2":      m_rank2,        # to show how negligible rank-2 penalty is
        "is_nash":      gain_up <= 0 and gain_down <= 0,
    }

def part_1_symmetric_nash():
    print("=" * 70)
    print("PART 1 — SYMMETRIC NASH CHECK")
    print(f"N = {N_TEAMS} teams")
    print("=" * 70)

    print(f"""
Key insight with N={N_TEAMS}:
  m(rank 1) = 0.9000
  m(rank 2) = {speed_multiplier_from_rank(2):.6f}   ← negligible difference
  m(rank N) = 0.1000

  Upward deviation (z*+1): costs 500 XIRECs, multiplier stays ≈ 0.9
  → almost never profitable

  Downward deviation (SP=0): saves budget, but m drops to 0.1
  → only profitable if research/scale gains outweigh the multiplier collapse
""")

    print(f"  {'z*':>4}  {'R*':>6}  {'SC*':>6}  {'PnL conform':>14}  "
          f"{'gain_up':>10}  {'gain_down':>12}  {'Nash?':>6}")
    print("  " + "─" * 68)

    nash_candidates = []
    for z_star in range(0, 101, 5):
        r = deviation_gain(z_star)
        tag = "✓" if r["is_nash"] else "✗"
        print(f"  {z_star:>4}  {r['r_conform']:>6.1f}  {r['sc_conform']:>6.1f}  "
              f"{r['pnl_conform']:>+14,.0f}  "
              f"{r['gain_up']:>+10,.0f}  {r['gain_down']:>+12,.0f}  {tag:>6}")
        if r["is_nash"]:
            nash_candidates.append(z_star)

    print(f"\n  Nash equilibria found at z* = {nash_candidates}")
    print(f"  Pareto-optimal Nash: z*={nash_candidates[0]} "
          f"(maximizes PnL, everyone better off)\n")

    return nash_candidates


# ═══════════════════════════════════════════════════════════════════════════════
# PART 2 — Best response dynamics with partial updating
# ═══════════════════════════════════════════════════════════════════════════════

def find_best_response_continuous(others_sp, dirichlet_alpha):
    """
    Find your best (R, SC, SP) given a fixed vector of others' speeds.
    Uses SLSQP — deterministic since others_sp is fixed.
    """
    def obj(x):
        r, sc, sp = x
        percentile  = np.mean(others_sp <= sp)
        multiplier  = 0.1 + 0.8 * percentile
        budget_used = BUDGET * (r + sc + sp) / 100
        return -(research(r) * scale(sc) * multiplier - budget_used)

    res = minimize(
        obj,
        x0=[16, 48, 35],
        method="SLSQP",
        bounds=[(0,100),(0,100),(0,100)],
        constraints=[{"type":"ineq","fun": lambda x: 100 - sum(x)}],
        options={"ftol":1e-9, "maxiter":500}
    )
    return res.x   # (r, sc, sp)

def part_2_best_response_dynamics(dirichlet_alpha, n_teams=500,
                                   n_iter=30, frac_update=0.1):
    """
    Cournot-style best response dynamics:
    at each step, a random fraction of teams updates to their best response.
    Partial updating (10%) is more realistic than synchronous full updating.
    Uses continuous SP values throughout.
    """
    print("=" * 70)
    print("PART 2 — BEST RESPONSE DYNAMICS (partial updating)")
    print(f"N={n_teams} teams, {frac_update*100:.0f}% update per round, "
          f"{n_iter} rounds")
    print("=" * 70)

    rng = np.random.default_rng(42)

    # initialize SP values from the mixture distribution
    proportions = np.random.dirichlet(dirichlet_alpha)
    pi_a, pi_b, pi_c = proportions
    n_a = int(pi_a * n_teams)
    n_b = int(pi_b * n_teams)
    n_c = n_teams - n_a - n_b
    field_sp = np.concatenate([
        sample_type_A(n_a),
        sample_type_B(n_b),
        sample_type_C(n_c, nash_sp=35.74),   # start from naive
    ])

    print(f"\n  Initial field: mean={field_sp.mean():.1f}  "
          f"median={np.median(field_sp):.1f}  "
          f"p25={np.percentile(field_sp,25):.1f}  "
          f"p75={np.percentile(field_sp,75):.1f}")
    print()

    trajectory = [field_sp.copy()]

    for t in range(n_iter):
        n_update = max(1, int(n_teams * frac_update))
        idx_update = rng.choice(n_teams, size=n_update, replace=False)

        for i in idx_update:
            others_sp = np.delete(field_sp, i)
            _, _, sp_br = find_best_response_continuous(others_sp, dirichlet_alpha)
            field_sp[i] = sp_br

        trajectory.append(field_sp.copy())

        if t in [0, 2, 5, 10, 20, n_iter-1]:
            print(f"  iter {t+1:>3}: mean={field_sp.mean():>5.1f}  "
                  f"median={np.median(field_sp):>5.1f}  "
                  f"p25={np.percentile(field_sp,25):>5.1f}  "
                  f"p75={np.percentile(field_sp,75):>5.1f}")

    return trajectory, field_sp


# ═══════════════════════════════════════════════════════════════════════════════
# PART 3 — CMA-ES Nash with heterogeneous population
# ═══════════════════════════════════════════════════════════════════════════════

def find_nash_cma(dirichlet_alpha, n_trials=2_000,
                  tol=0.5, max_iter=100):
    """
    Fixed point iteration over rational SP using CMA-ES.

    At each candidate SP_candidate:
      - Type A and B play their fixed distributions
      - Type C plays SP_candidate  ← breaks circular dependency
      - CMA-ES finds your best (R, SC, SP) against this population
      - your optimal SP becomes the next SP_candidate
      - repeat until convergence

    CMA-ES used instead of SLSQP because:
      - objective is stochastic (MC over Dirichlet draws)
      - SLSQP's finite-difference gradients are pure noise in this setting
    """
    sp_candidate = 35.74
    history = []

    for i in range(max_iter):

        def obj(x):
            r, sc, sp = x
            if r < 0 or sc < 0 or sp < 0 or r + sc + sp > 100:
                return 1e9
            pnls = []
            for _ in range(n_trials):
                proportions = np.random.dirichlet(dirichlet_alpha)
                percentile  = mixture_cdf_with_rational_sp(
                                sp, sp_candidate, proportions)
                multiplier  = 0.1 + 0.8 * percentile
                budget_used = BUDGET * (r + sc + sp) / 100
                pnls.append(research(r) * scale(sc) * multiplier - budget_used)
            return -np.mean(pnls)

        es = cma.CMAEvolutionStrategy(
            x0=[16, 48, sp_candidate],
            sigma0=5,
            inopts={
                "bounds":   [[0,0,0],[100,100,100]],
                "tolx":     0.1,
                "tolfun":   10,
                "maxiter":  100,
                "verbose":  -9
            }
        )
        es.optimize(obj)
        r_new, sc_new, sp_new = es.result.xbest

        total = r_new + sc_new + sp_new
        if total > 100:
            r_new  *= 100 / total
            sc_new *= 100 / total
            sp_new *= 100 / total

        delta = abs(sp_new - sp_candidate)
        print(f"  iter {i+1:3d} | R={r_new:.2f}%  SC={sc_new:.2f}%  "
              f"SP={sp_new:.2f}% | delta={delta:.4f}")

        history.append({"iter": i+1, "r": r_new, "sc": sc_new,
                        "sp": sp_new, "delta": delta})

        if delta < tol:
            print(f"\n  Converged in {i+1} iterations — Nash SP* = {sp_new:.2f}%")
            break

        sp_candidate = sp_new
    else:
        print(f"\n  Warning: did not converge in {max_iter} iterations")
        print(f"  Last SP = {sp_new:.2f}%  delta = {delta:.4f}")

    return pd.DataFrame(history), (r_new, sc_new, sp_new)


# ═══════════════════════════════════════════════════════════════════════════════
# PART 4 — Stability verification
# ═══════════════════════════════════════════════════════════════════════════════

def verify_stability(r_star, sc_star, sp_star, dirichlet_alpha,
                     n_trials=2_000,
                     deviation_grid=np.linspace(-20, 20, 41)):
    """
    Fix others at Nash SP*, sweep your own SP around sp_star.
    If sp_star is a true Nash equilibrium, PnL should peak at sp_star.
    R and SC are re-optimized at each deviation point.
    """
    print("\n  Stability check: sweeping SP deviation from Nash SP*...")
    records = []

    for delta_sp in deviation_grid:
        sp_dev = np.clip(sp_star + delta_sp, 0, 100)

        pnls = []
        for _ in range(n_trials):
            proportions = np.random.dirichlet(dirichlet_alpha)
            # others fixed at nash sp_star, only you deviate
            percentile  = mixture_cdf_with_rational_sp(
                            sp_dev, sp_star, proportions)
            multiplier  = 0.1 + 0.8 * percentile

            # re-optimize R/SC given your SP deviation
            pnl_rc, _   = optimize_r_sc(sp_dev, multiplier)
            pnls.append(pnl_rc)

        records.append({
            "delta_sp": delta_sp,
            "sp":       sp_dev,
            "pnl":      np.mean(pnls),
            "std":      np.std(pnls)
        })

    df = pd.DataFrame(records)
    peak_sp = df.loc[df["pnl"].idxmax(), "sp"]
    print(f"  PnL peaks at SP={peak_sp:.1f}%  (Nash SP*={sp_star:.1f}%)")
    if abs(peak_sp - sp_star) < 2:
        print("  ✓ Confirmed Nash equilibrium — no profitable deviation")
    else:
        print(f"  ✗ Peak SP differs from Nash SP* by {abs(peak_sp-sp_star):.1f}pp "
              f"— equilibrium not confirmed")
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN — run all parts
# ═══════════════════════════════════════════════════════════════════════════════

scenarios = {
    "uniform belief":   [1,  1,  1],
    "naive dominated":  [20, 5,  1],
    "nash dominated":   [20, 5, 75],
    "nash dominated_2": [10, 10, 80],
    "nash dominated_3": [5,  5, 90],
}

# ── Part 1: always run first — tells you whether unique Nash exists ────────────
nash_candidates = part_1_symmetric_nash()

# ── Parts 2, 3, 4: run per scenario ───────────────────────────────────────────
nash_results = {}

for name, alpha in scenarios.items():
    print(f"\n{'=' * 70}")
    print(f"Scenario: {name}   alpha={alpha}")
    print(f"{'=' * 70}")

    # Part 2: best response dynamics
    trajectory, final_field = part_2_best_response_dynamics(
        alpha, n_teams=500, n_iter=30, frac_update=0.1
    )
    print(f"\n  BR dynamics settled at: "
          f"mean SP={final_field.mean():.1f}%  "
          f"median SP={np.median(final_field):.1f}%")

    # Part 3: CMA-ES Nash
    print(f"\n  CMA-ES Nash iteration:")
    history_df, (r_star, sc_star, sp_star) = find_nash_cma(
        dirichlet_alpha=alpha,
        n_trials=2_000,
        tol=0.5,
        max_iter=100
    )

    # Part 4: stability
    stab_df = verify_stability(r_star, sc_star, sp_star, alpha, n_trials=2_000)

    nash_results[name] = {
        "r":           r_star,
        "sc":          sc_star,
        "sp":          sp_star,
        "history":     history_df,
        "stability":   stab_df,
        "br_dynamics": final_field,
    }

    print(f"\n  Nash equilibrium for '{name}':")
    print(f"    R*     = {r_star:.2f}%")
    print(f"    SC*    = {sc_star:.2f}%")
    print(f"    SP*    = {sp_star:.2f}%")
    print(f"    Total  = {r_star+sc_star+sp_star:.2f}%")

PART 1 — SYMMETRIC NASH CHECK
N = 20000 teams

Key insight with N=20000:
  m(rank 1) = 0.9000
  m(rank 2) = 0.899960   ← negligible difference
  m(rank N) = 0.1000

  Upward deviation (z*+1): costs 500 XIRECs, multiplier stays ≈ 0.9
  → almost never profitable

  Downward deviation (SP=0): saves budget, but m drops to 0.1
  → only profitable if research/scale gains outweigh the multiplier collapse

    z*      R*     SC*     PnL conform     gain_up     gain_down   Nash?
  ────────────────────────────────────────────────────────────────────
     0    23.1    76.9        +618,111      -8,688      -593,878       ✓
     5    22.2    72.8        +574,915      -8,564      -550,681       ✓
    10    21.2    68.8        +532,305      -8,451      -508,071       ✓
    15    20.2    64.8        +490,299      -8,138      -466,066       ✓
    20    19.2    60.8        +448,942      -8,208      -424,709       ✓
    25    18.2    56.8        +408,235      -8,062      -384,002       ✓
    30    17.2  

PART 1 — SYMMETRIC NASH CHECK
N = 20000 teams

Key insight with N=20000:
  m(rank 1) = 0.9000
  m(rank 2) = 0.899960   ← negligible difference
  m(rank N) = 0.1000

  Upward deviation (z*+1): costs 500 XIRECs, multiplier stays ≈ 0.9
  → almost never profitable

  Downward deviation (SP=0): saves budget, but m drops to 0.1
  → only profitable if research/scale gains outweigh the multiplier collapse

    z*      R*     SC*     PnL conform     gain_up     gain_down   Nash?
  ────────────────────────────────────────────────────────────────────
     0    23.1    76.9        +618,111      -8,688      -593,878       ✓
     5    22.2    72.8        +574,915      -8,564      -550,681       ✓
    10    21.2    68.8        +532,305      -8,451      -508,071       ✓
    15    20.2    64.8        +490,299      -8,138      -466,066       ✓
    20    19.2    60.8        +448,942      -8,208      -424,709       ✓
    25    18.2    56.8        +408,235      -8,062      -384,002       ✓
    30    17.2  

KeyboardInterrupt: 

In [26]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
import cma

BUDGET  = 50_000
N_TEAMS = 20_000

def research(x):
    return 200_000 * np.log(1 + x) / np.log(1 + 100)

def scale(x):
    return 7 * x / 100

def speed_multiplier_from_rank(rank, n=N_TEAMS):
    return 0.9 - 0.8 * (rank - 1) / (n - 1)

def mixture_cdf_analytical(speed_val, rational_sp, proportions):
    """Exact mixture CDF — replaces all Monte Carlo population sampling."""
    pi_a, pi_b, pi_c = proportions
    cdf_a = norm.cdf(speed_val, loc=35.74, scale=4.0)
    cdf_b = np.clip(speed_val / 100, 0, 1)
    cdf_c = norm.cdf(speed_val, loc=rational_sp, scale=2.0)
    return pi_a * cdf_a + pi_b * cdf_b + pi_c * cdf_c

def optimize_r_sc(sp_pct, multiplier):
    """Optimize R and SC given fixed SP and multiplier — deterministic, SLSQP fine."""
    def obj(x):
        r, sc = x
        budget_used = BUDGET * (r + sc + sp_pct) / 100
        return -(research(r) * scale(sc) * multiplier - budget_used)

    res = minimize(
        obj, x0=[16, 48], method="SLSQP",
        bounds=[(0, 100-sp_pct), (0, 100-sp_pct)],
        constraints=[{"type":"ineq","fun": lambda x: (100-sp_pct) - sum(x)}],
        options={"ftol":1e-12}
    )
    return -res.fun, res.x


# ═══════════════════════════════════════════════════════════════════════════════
# PART 1 — Symmetric Nash check
# ═══════════════════════════════════════════════════════════════════════════════

def deviation_gain(z_star, n=N_TEAMS):
    pnl_conform, (r_c, sc_c) = optimize_r_sc(z_star, multiplier=0.9)
    pnl_up,   _              = optimize_r_sc(min(z_star+1, 100), multiplier=0.9) if z_star < 100 else (pnl_conform, None)
    pnl_down, _              = optimize_r_sc(0, multiplier=0.1)
    return {
        "z_star":      z_star,
        "pnl_conform": pnl_conform,
        "r_conform":   r_c,
        "sc_conform":  sc_c,
        "gain_up":     pnl_up   - pnl_conform,
        "gain_down":   pnl_down - pnl_conform,
        "m_rank2":     speed_multiplier_from_rank(2, n),
        "is_nash":     (pnl_up - pnl_conform) <= 0 and (pnl_down - pnl_conform) <= 0,
    }

def part_1_symmetric_nash():
    print("=" * 70)
    print(f"PART 1 — SYMMETRIC NASH CHECK  (N={N_TEAMS})")
    print(f"  m(rank 2) = {speed_multiplier_from_rank(2):.6f}  ← upward deviation gain ≈ 0")
    print("=" * 70)
    print(f"\n  {'z*':>4}  {'R*':>6}  {'SC*':>6}  {'PnL conform':>14}  "
          f"{'gain_up':>10}  {'gain_down':>12}  {'Nash?':>6}")
    print("  " + "─" * 68)

    nash_candidates = []
    for z_star in range(0, 101, 5):
        r = deviation_gain(z_star)
        tag = "✓" if r["is_nash"] else "✗"
        print(f"  {z_star:>4}  {r['r_conform']:>6.1f}  {r['sc_conform']:>6.1f}  "
              f"{r['pnl_conform']:>+14,.0f}  "
              f"{r['gain_up']:>+10,.0f}  {r['gain_down']:>+12,.0f}  {tag:>6}")
        if r["is_nash"]:
            nash_candidates.append(z_star)

    print(f"\n  Nash equilibria at z* = {nash_candidates}")
    print(f"  Pareto-optimal Nash:  z* = {nash_candidates[0] if nash_candidates else 'none'}\n")
    return nash_candidates


# ═══════════════════════════════════════════════════════════════════════════════
# PART 2 — Best response dynamics with partial updating
# ═══════════════════════════════════════════════════════════════════════════════

def find_best_response_continuous(others_sp):
    """Deterministic best response — others_sp is fixed, no noise."""
    def obj(x):
        r, sc, sp = x
        percentile  = np.mean(others_sp <= sp)
        multiplier  = 0.1 + 0.8 * percentile
        budget_used = BUDGET * (r + sc + sp) / 100
        return -(research(r) * scale(sc) * multiplier - budget_used)

    res = minimize(
        obj, x0=[16, 48, 35], method="SLSQP",
        bounds=[(0,100),(0,100),(0,100)],
        constraints=[{"type":"ineq","fun": lambda x: 100 - sum(x)}],
        options={"ftol":1e-9, "maxiter":500}
    )
    return res.x

def part_2_best_response_dynamics(dirichlet_alpha, n_teams=500,
                                   n_iter=30, frac_update=0.1):
    print("=" * 70)
    print(f"PART 2 — BEST RESPONSE DYNAMICS  "
          f"(N={n_teams}, {frac_update*100:.0f}%/round, {n_iter} rounds)")
    print("=" * 70)

    rng = np.random.default_rng(42)
    proportions = np.random.dirichlet(dirichlet_alpha)
    pi_a, pi_b, pi_c = proportions
    n_a = int(pi_a * n_teams)
    n_b = int(pi_b * n_teams)
    n_c = n_teams - n_a - n_b

    field_sp = np.concatenate([
        np.clip(np.random.normal(35.74, 4.0, n_a), 0, 100),
        np.random.uniform(0, 100, n_b),
        np.clip(np.random.normal(35.74, 2.0, n_c), 0, 100),
    ])

    for t in range(n_iter):
        n_update  = max(1, int(n_teams * frac_update))
        idx_update = rng.choice(n_teams, size=n_update, replace=False)
        for i in idx_update:
            others_sp    = np.delete(field_sp, i)
            _, _, sp_br  = find_best_response_continuous(others_sp)
            field_sp[i]  = sp_br

        if t in [0, 2, 5, 10, 20, n_iter-1]:
            print(f"  iter {t+1:>3}: mean={field_sp.mean():>5.1f}  "
                  f"median={np.median(field_sp):>5.1f}  "
                  f"p25={np.percentile(field_sp,25):>5.1f}  "
                  f"p75={np.percentile(field_sp,75):>5.1f}")

    print(f"\n  Settled: mean SP={field_sp.mean():.1f}%  "
          f"median SP={np.median(field_sp):.1f}%\n")
    return field_sp


# ═══════════════════════════════════════════════════════════════════════════════
# PART 3 — CMA-ES Nash with heterogeneous population (fast version)
# ═══════════════════════════════════════════════════════════════════════════════

def find_nash_cma(dirichlet_alpha, n_trials=2_000,
                  tol=0.5, max_iter=100):
    """
    Fixed point iteration using CMA-ES + analytical CDF + vectorized MC.
    Speedups vs previous version:
      1. mixture_cdf_analytical replaces 20k-sample Monte Carlo per trial
      2. all n_trials computed in one vectorized numpy pass
      3. no Python for loop inside the objective
    """
    sp_candidate = 35.74
    history = []

    for i in range(max_iter):

        def obj(x):
            r, sc, sp = x
            if r < 0 or sc < 0 or sp < 0 or r + sc + sp > 100:
                return 1e9

            # draw all proportions at once
            all_props = np.random.dirichlet(dirichlet_alpha, size=n_trials)
            pi_a = all_props[:, 0]
            pi_b = all_props[:, 1]
            pi_c = all_props[:, 2]

            # analytical CDF — scalars, no sampling
            cdf_a = norm.cdf(sp, loc=35.74,        scale=4.0)
            cdf_b = np.clip(sp / 100, 0, 1)
            cdf_c = norm.cdf(sp, loc=sp_candidate,  scale=2.0)

            # vectorized over all trials
            percentiles = pi_a * cdf_a + pi_b * cdf_b + pi_c * cdf_c
            multipliers = 0.1 + 0.8 * percentiles
            budget_used = BUDGET * (r + sc + sp) / 100
            pnls        = research(r) * scale(sc) * multipliers - budget_used

            return -np.mean(pnls)

        es = cma.CMAEvolutionStrategy(
            x0=[16, 48, sp_candidate],
            sigma0=5,
            inopts={
                "bounds":  [[0,0,0],[100,100,100]],
                "tolx":    0.1,
                "tolfun":  10,
                "maxiter": 100,
                "verbose": -9
            }
        )
        es.optimize(obj)
        r_new, sc_new, sp_new = es.result.xbest

        total = r_new + sc_new + sp_new
        if total > 100:
            r_new  *= 100 / total
            sc_new *= 100 / total
            sp_new *= 100 / total

        delta = abs(sp_new - sp_candidate)
        print(f"  iter {i+1:3d} | R={r_new:.2f}%  SC={sc_new:.2f}%  "
              f"SP={sp_new:.2f}% | delta={delta:.4f}")

        history.append({"iter": i+1, "r": r_new, "sc": sc_new,
                        "sp": sp_new, "delta": delta})

        if delta < tol:
            print(f"\n  Converged in {i+1} iterations — Nash SP* = {sp_new:.2f}%")
            break

        sp_candidate = sp_new
    else:
        print(f"\n  Warning: did not converge in {max_iter} iterations")

    return pd.DataFrame(history), (r_new, sc_new, sp_new)


# ═══════════════════════════════════════════════════════════════════════════════
# PART 4 — Stability verification
# ═══════════════════════════════════════════════════════════════════════════════

def verify_stability(r_star, sc_star, sp_star, dirichlet_alpha,
                     n_trials=2_000,
                     deviation_grid=np.linspace(-20, 20, 41)):
    print("\n  Stability check...")
    records = []

    for delta_sp in deviation_grid:
        sp_dev = np.clip(sp_star + delta_sp, 0, 100)

        # vectorized over trials
        all_props = np.random.dirichlet(dirichlet_alpha, size=n_trials)
        pi_a = all_props[:, 0]
        pi_b = all_props[:, 1]
        pi_c = all_props[:, 2]

        cdf_a = norm.cdf(sp_dev, loc=35.74,   scale=4.0)
        cdf_b = np.clip(sp_dev / 100, 0, 1)
        cdf_c = norm.cdf(sp_dev, loc=sp_star,  scale=2.0)

        percentiles = pi_a * cdf_a + pi_b * cdf_b + pi_c * cdf_c
        multipliers = 0.1 + 0.8 * percentiles

        # re-optimize R/SC at mean multiplier (deterministic)
        mean_mult       = np.mean(multipliers)
        pnl_rc, _       = optimize_r_sc(sp_dev, mean_mult)

        records.append({
            "delta_sp": delta_sp,
            "sp":       sp_dev,
            "pnl":      pnl_rc,
        })

    df      = pd.DataFrame(records)
    peak_sp = df.loc[df["pnl"].idxmax(), "sp"]
    print(f"  PnL peaks at SP={peak_sp:.1f}%  (Nash SP*={sp_star:.1f}%)")
    if abs(peak_sp - sp_star) < 2:
        print("  ✓ Confirmed Nash equilibrium")
    else:
        print(f"  ✗ Peak differs by {abs(peak_sp-sp_star):.1f}pp — not confirmed")
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

scenarios = {
    # "uniform belief":   [1,  1,  1],
    # "naive dominated":  [20, 5,  1],
    "nash dominated":   [20, 5, 75],
    "nash dominated_2": [10, 10, 80],
    "nash dominated_3": [5,  5, 90],
}

nash_candidates = part_1_symmetric_nash()

nash_results = {}
for name, alpha in scenarios.items():
    print(f"\n{'='*70}")
    print(f"Scenario: {name}   alpha={alpha}")
    print(f"{'='*70}")

    final_field = part_2_best_response_dynamics(
        alpha, n_teams=500, n_iter=30, frac_update=0.1)

    print(f"\n  CMA-ES Nash iteration:")
    history_df, (r_star, sc_star, sp_star) = find_nash_cma(
        dirichlet_alpha=alpha, n_trials=2_000, tol=0.5, max_iter=100)

    stab_df = verify_stability(r_star, sc_star, sp_star, alpha, n_trials=2_000)

    nash_results[name] = {
        "r": r_star, "sc": sc_star, "sp": sp_star,
        "history": history_df, "stability": stab_df,
    }

    print(f"\n  Nash equilibrium for '{name}':")
    print(f"    R* = {r_star:.2f}%  SC* = {sc_star:.2f}%  SP* = {sp_star:.2f}%")
    print(f"    Total = {r_star+sc_star+sp_star:.2f}%")

PART 1 — SYMMETRIC NASH CHECK  (N=20000)
  m(rank 2) = 0.899960  ← upward deviation gain ≈ 0

    z*      R*     SC*     PnL conform     gain_up     gain_down   Nash?
  ────────────────────────────────────────────────────────────────────
     0    23.1    76.9        +618,111      -8,688      -593,878       ✓
     5    22.2    72.8        +574,915      -8,564      -550,681       ✓
    10    21.2    68.8        +532,305      -8,451      -508,071       ✓
    15    20.2    64.8        +490,299      -8,138      -466,066       ✓
    20    19.2    60.8        +448,942      -8,208      -424,709       ✓
    25    18.2    56.8        +408,235      -8,062      -384,002       ✓
    30    17.2    52.8        +368,252      -7,912      -344,019       ✓
    35    16.2    48.8        +329,031      -7,745      -304,798       ✓
    40    15.1    44.9        +290,676      -7,602      -266,443       ✓
    45    14.1    40.9        +253,134      -7,390      -228,900       ✓
    50    13.0    37.0        +2

# The right framework is a Bayesian mixture model with Monte Carlo integration


### Type definitions — A, B, C

In [ ]:
# Dirichlet(alpha) controls how concentrated vs diffuse the mixture is
# alpha = [1,1,1] → completely uninformed (uniform over all mixtures)
# alpha = [5,3,1] → you believe naive players dominate
# alpha = [2,2,2] → mild belief in symmetry


def sample_type_A(n):
    """Naive: ran direct-mapping optimizer, some noise around 35.74%"""
    return np.clip(np.random.normal(loc=35.74, scale=4.0, size=n), 0, 100)

def sample_type_B(n):
    """Uninformed rational: know it's rank-based but no belief about others.
    Uniform is the maximum entropy prior — most honest when you know nothing."""
    return np.random.uniform(10, 90, size=n)

def sample_type_C(n):
    """Coordinated rational: converged toward Nash, tight cluster.
    We use the Nash speed we found earlier as the mean."""
    nash_sp = 33.33
    return np.clip(np.random.normal(loc=nash_sp, scale=2.0, size=n), 0, 100)


Given a specific mixture composition (e.g. 20% A, 5% B, 75% C), it synthetically generates 50,000 players by sampling from each type proportionally, then asks: what fraction of that population has speed ≤ your speed? This fraction is your percentile. With 50,000 samples it's a very stable estimate of the true CDF of the mixture distribution.


Converts your percentile directly into your speed multiplier. If you're at the 80th percentile of the population, you get 0.1 + 0.8 * 0.8 = 0.74. This replaces speed_real entirely — instead of dense-ranking among 9 players, you're locating yourself in a continuous population distribution.


In [ ]:
def mixture_cdf(speed_val, proportions, n_samples=20_000):
    """
    Estimate P(opponent_speed <= speed_val) from the mixture.
    With 20k players this CDF is essentially deterministic.
    """

    # 1. figure out how many players of each type to generate
    pi_a, pi_b, pi_c = proportions
    n_a = int(pi_a * n_samples)
    n_b = int(pi_b * n_samples)
    n_c = n_samples - n_a - n_b

    # 2. generate each sub-population's speed allocations
    # 3. pool them into one population of exactly 20,000
    population = np.concatenate([
        sample_type_A(n_a),
        sample_type_B(n_b),
        sample_type_C(n_c),
    ])
    # 4. ask: what fraction of the population is at or below my speed?
    return np.mean(population <= speed_val) # e.g. if speed_val=40 and 14,000/20,000 players are below 40 → returns 0.70

def mixture_cdf_analytical(speed_val, rational_sp, proportions):
    """
    Exact mixture CDF — no Monte Carlo sampling.
    F(x) = pi_a * CDF_A(x) + pi_b * CDF_B(x) + pi_c * CDF_C(x)

    Type A ~ Normal(35.74, 4)       naive players
    Type B ~ Uniform(0, 100)        uninformed rational
    Type C ~ Normal(rational_sp, 2) coordinated rational
    """
    pi_a, pi_b, pi_c = proportions
    cdf_a = norm.cdf(speed_val, loc=35.74,      scale=4.0)
    cdf_b = np.clip(speed_val / 100, 0, 1)
    cdf_c = norm.cdf(speed_val, loc=rational_sp, scale=2.0)
    return pi_a * cdf_a + pi_b * cdf_b + pi_c * cdf_c


def speed_global_rank(my_sp, proportions, n_samples=20_000):
    """
    With a global ranking of 20k teams:
    - your percentile in the population = fraction of players at or below you
    - rank 1 (top) → multiplier 0.9
    - rank last (bottom) → multiplier 0.1
    - linear interpolation in between

    percentile=1.0 means everyone is below you → you're rank 1 → 0.9
    percentile=0.0 means everyone is above you → you're last → 0.1
    """
    percentile = mixture_cdf(my_sp, proportions, n_samples)
    return 0.1 + 0.8 * percentile

def PnL_global(r_pct, sc_pct, sp_pct, proportions, pop_samples=20_000):
    total_alloc = r_pct + sc_pct + sp_pct
    budget_used = BUDGET * (total_alloc / 100)
    gross = research(r_pct) * scale(sc_pct) * speed_global_rank(sp_pct, proportions, pop_samples)
    return gross - budget_used


`monte_carlo_expected_pnl_global(...)`
This is where the two layers of uncertainty are integrated:

- Outer loop (n_trials iterations): draws a fresh mixture composition from the Dirichlet each trial — so one trial might get `[0.18, 0.06, 0.76]`, the next `[0.22, 0.04, 0.74]`, etc. This represents your uncertainty about the true composition of the 20,000-team field.
- Inner (inside each trial): given that composition, generates the population CDF and computes your percentile deterministically.

The output — mean, std, p10, p90 — describes the full distribution of your PnL outcomes across all possible field compositions consistent with your Dirichlet belief.


In [5]:
def monte_carlo_expected_pnl_global(r_pct, sc_pct, sp_pct,
                                     dirichlet_alpha,
                                     n_trials=2_000,
                                     pop_samples=20_000):
    """
    Outer loop: uncertainty over mixture proportions (Dirichlet)
    Inner: compute your percentile in the population drawn from that mixture
    """
    pnls = []
    for _ in range(n_trials):
        proportions = np.random.dirichlet(dirichlet_alpha)
        pnl = PnL_global(r_pct, sc_pct, sp_pct, proportions, pop_samples)
        pnls.append(pnl)

    return {
        "mean": np.mean(pnls),
        "std":  np.std(pnls),
        "p10":  np.percentile(pnls, 10),
        "p90":  np.percentile(pnls, 90),
        "pnls": pnls
    }

`optimize_global(dirichlet_alpha)`

Runs SLSQP to find the (R, SC, SP) triple that maximizes expected PnL under the Monte Carlo model. The stochastic objective is why ftol=1e-4 is used instead of 1e-12 — tightening the tolerance would just chase noise in the Monte Carlo estimate rather than finding a better solution.


In [8]:
def optimize_global(dirichlet_alpha, n_trials=1_000):
    def obj(x):
        result = monte_carlo_expected_pnl_global(
            x[0], x[1], x[2],
            dirichlet_alpha=dirichlet_alpha,
            n_trials=n_trials
        )
        return -result["mean"]

    res = minimize(
        obj,
        x0=[16, 48, 36],
        method="SLSQP",
        bounds=[(0,100),(0,100),(0,100)],
        constraints=[{"type":"ineq","fun": lambda x: 100 - sum(x)}],
        options={"ftol":1e-4, "maxiter":2000}
    )
    return res

**The scenario loop**

Runs the full pipeline three times under three different beliefs about the field composition. The Dirichlet alphas translate to these expected proportions:

| Scenario | E[Type A naive] | E[Type B uniform] | E[Type C Nash] |
|---|---|---|---|
| uniform belief | 33% | 33% | 33% |
| naive dominated | 77% | 19% | 4% |
| nash dominated | 20% | 5% | 75% |

The output tells you: given your belief about who you're competing against, what should your optimal (R, SC, SP) be?

In [ ]:
# your three scenarios
scenarios = {
    # "uniform belief":        [1,  1,  1],
    # "naive dominated":       [20, 5,  1],
    # "nash dominated":        [20, 5, 75],
    "nash dominated_2":      [10, 10, 80],
    "nash dominated_3":      [5, 5, 90],
}

nash_sp_per_scenario = {
    "uniform belief":   None,          # did not converge — use 42% as midpoint estimate
    "naive dominated":  43.23 + 4,     # → 47%
    "nash dominated":   45.92 + 4,     # → 50%
    "nash dominated_2": 39.65 + 4,     # → 44%
    "nash dominated_3": 46.39 + 4,     # → 50%
}


# for name, alpha in scenarios.items():
#     res = optimize_global(alpha)
#     r, sc, sp = res.x
#     print(f"\n{name}  (alpha={alpha})")
#     print(f"  R={r:.1f}%  SC={sc:.1f}%  SP={sp:.1f}%  E[PnL]={-res.fun:,.0f}")

BUDGET = 50_000

for name, alpha in scenarios.items():
    # step 1: find optimal allocation
    res = optimize_global(alpha, n_trials=2_000)
    r_opt, sc_opt, sp_opt = res.x
    
    # step 2: re-evaluate at the optimum with more trials to get clean distribution
    result = monte_carlo_expected_pnl_global(
        r_opt, sc_opt, sp_opt,
        dirichlet_alpha=alpha,
        n_trials=5_000,
        pop_samples=20_000
    )
    
    print(f"\n{name}  (alpha={alpha})")
    print(f"  Optimal: R={r_opt:.1f}%  SC={sc_opt:.1f}%  SP={sp_opt:.1f}%")
    print(f"  E[PnL]  = {result['mean']:>12,.0f}")
    print(f"  Std     = {result['std']:>12,.0f}")
    print(f"  P10     = {result['p10']:>12,.0f}")   # bad day
    print(f"  P90     = {result['p90']:>12,.0f}")   # good day


uniform belief  (alpha=[1, 1, 1])
  Optimal: R=16.0%  SC=48.0%  SP=36.0%
  E[PnL]  =      188,149
  Std     =       37,731
  P10     =      141,532
  P90     =      243,152

naive dominated  (alpha=[20, 5, 1])
  Optimal: R=16.0%  SC=48.0%  SP=36.0%
  E[PnL]  =      159,157
  Std     =        6,650
  P10     =      151,400
  P90     =      167,491

nash dominated  (alpha=[20, 5, 75])
  Optimal: R=16.0%  SC=48.0%  SP=36.0%
  E[PnL]  =      256,843
  Std     =        6,123
  P10     =      248,761
  P90     =      264,406

nash dominated_2  (alpha=[10, 10, 80])
  Optimal: R=16.0%  SC=48.0%  SP=36.0%
  E[PnL]  =      260,547
  Std     =        6,280
  P10     =      252,259
  P90     =      268,068


```bash 
FOR each scenario (uniform / naive_dominated / nash_dominated):
│
│   alpha = Dirichlet hyperparameter for this scenario
│
│   ┌─ OPTIMIZATION PHASE (n_trials=2000) ─────────────────────────────────────┐
│   │                                                                          │
│   │      SLSQP searches over (R, SC, SP) to minimize -E[PnL]                 │
│   │                                                                          │
│   │      SLSQP function evaluation at candidate (R, SC, SP),                 │
│   │      WHILE not converged (up to maxiter=200):                            │
│   │      │                                                                   │
│   │      │                                                                   │
│   │      │        monte_carlo_expected_pnl_global(...)                       │
│   │      │        │                                                          │
│   │      │        FOR trial in 1..2000:                                      │
│   │      │        │                                                          │
│   │      │        │   1. DRAW proportions ~ Dirichlet(alpha)                 │
│   │      │        │      → e.g. [pi_A=0.19, pi_B=0.06, pi_C=0.75]            │
│   │      │        │                                                          │
│   │      │        │   2. GENERATE population of 20,000 speeds:               │
│   │      │        │      - 0.19 * 20000 = 3800  players ~ Normal(35.74, 4)   │
│   │      │        │      - 0.06 * 20000 = 1200  players ~ Uniform(0, 100)    │
│   │      │        │      - 0.75 * 20000 = 15000 players ~ Normal(33.33, 2)   │
│   │      │        │                                                          │
│   │      │        │   3. COMPUTE percentile = mean(population <= my_sp)      │
│   │      │        │                                                          │
│   │      │        │   4. COMPUTE speed_multiplier = 0.1 + 0.8 * percentile   │
│   │      │        │                                                          │
│   │      │        │   5. COMPUTE PnL = research(R) * scale(SC)               │
│   │      │        │                  * speed_multiplier                      │
│   │      │        │                  - BUDGET * (R+SC+SP)/100                │
│   │      │        │                                                          │
│   │      │        └── collect PnL into list                                  │
│   │      │                                                                   │
│   │      │   RETURN -mean(PnL list)   ← SLSQP minimizes this                 │
│   │      │                                                                   │
│   │      └── SLSQP converges → optimal (R*, SC*, SP*)                        │
│   └──────────────────────────────────────────────────────────────────────────┘
│
│   ┌─ EVALUATION PHASE (n_trials=5000) ───────────────────────────────┐
│   │                                                                  │
│   │   SAME inner loop as above but:                                  │
│   │   - fixed at (R*, SC*, SP*)  — no optimization                   │
│   │   - 5000 trials for cleaner distribution estimate                │
│   │   - collect full PnL distribution                                │
│   │   - compute mean, std, P10, P90                                  │
│   │                                                                  │
│   └──────────────────────────────────────────────────────────────────┘
│
└── PRINT optimal allocation + PnL distribution for this scenario

```

*Notes:* What you should NOT do is generate multiple populations of 20,000 inside mixture_cdf itself and average them — that would cancel out the population variance that the outer loop is specifically trying to capture. The variance across trials is meaningful: it represents the real uncertainty you face about what the field looks like on competition day.

The difference is subtle but real
What we do now: each trial generates ONE population of 20,000, computes your percentile in it, gets one PnL. The variance across trials comes from two sources simultaneously:

- different proportion draws from the Dirichlet
- different random populations given those proportions

What averaging inside `mixture_cdf` would do: generate say 10 populations of 20,000 per trial and average the percentile before computing PnL. This would kill source 2 entirely — you'd only preserve variance from the Dirichlet draw. The problem is that on competition day, there is only ONE realization of the 20,000-player field. The randomness of that one draw matters and should be preserved in your distribution of outcomes.


In [28]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
import cma

BUDGET  = 50_000
N_TEAMS = 20_000

def research(x):
    return 200_000 * np.log(1 + x) / np.log(1 + 100)

def scale(x):
    return 7 * x / 100

# ── analytical mixture CDF ─────────────────────────────────────────────────────

def mixture_cdf_analytical(speed_val, rational_sp, proportions):
    """
    Exact mixture CDF — no Monte Carlo sampling.
    F(x) = pi_a * CDF_A(x) + pi_b * CDF_B(x) + pi_c * CDF_C(x)

    Type A ~ Normal(35.74, 4)       naive players
    Type B ~ Uniform(0, 100)        uninformed rational
    Type C ~ Normal(rational_sp, 2) coordinated rational
    """
    pi_a, pi_b, pi_c = proportions
    cdf_a = norm.cdf(speed_val, loc=35.74,      scale=4.0)
    cdf_b = np.clip(speed_val / 100, 0, 1)
    cdf_c = norm.cdf(speed_val, loc=rational_sp, scale=2.0)
    return pi_a * cdf_a + pi_b * cdf_b + pi_c * cdf_c

# ── vectorized Monte Carlo PnL ─────────────────────────────────────────────────

def monte_carlo_expected_pnl(r, sc, sp, rational_sp,
                              dirichlet_alpha, n_trials=2_000):
    """
    Vectorized over all n_trials — no Python loop.

    Two layers of uncertainty:
      outer: mixture proportions ~ Dirichlet(alpha)  [who is in the field]
      inner: analytical CDF given those proportions   [where you rank]
    """
    # draw all proportions at once: shape (n_trials, 3)
    all_props = np.random.dirichlet(dirichlet_alpha, size=n_trials)
    pi_a = all_props[:, 0]
    pi_b = all_props[:, 1]
    pi_c = all_props[:, 2]

    # analytical CDF — three scalars, broadcast over all trials
    cdf_a = norm.cdf(sp, loc=35.74,       scale=4.0)
    cdf_b = np.clip(sp / 100, 0, 1)
    cdf_c = norm.cdf(sp, loc=rational_sp,  scale=2.0)

    # percentile and multiplier for each trial
    percentiles = pi_a * cdf_a + pi_b * cdf_b + pi_c * cdf_c   # (n_trials,)
    multipliers = 0.1 + 0.8 * percentiles                        # (n_trials,)

    budget_used = BUDGET * (r + sc + sp) / 100
    pnls        = research(r) * scale(sc) * multipliers - budget_used

    return {
        "mean": np.mean(pnls),
        "std":  np.std(pnls),
        "p10":  np.percentile(pnls, 10),
        "p90":  np.percentile(pnls, 90),
        "pnls": pnls,
    }

# ── CMA-ES optimizer ───────────────────────────────────────────────────────────

def optimize_cma(dirichlet_alpha, rational_sp, n_trials=2_000):
    """
    Find optimal (R, SC, SP) using CMA-ES.
    Replaces SLSQP which fails on stochastic objectives.
    rational_sp: the Nash SP* for Type C players in this scenario.
    """
    def obj(x):
        r, sc, sp = x
        if r < 0 or sc < 0 or sp < 0 or r + sc + sp > 100:
            return 1e9
        result = monte_carlo_expected_pnl(
            r, sc, sp, rational_sp, dirichlet_alpha, n_trials
        )
        return -result["mean"]

    es = cma.CMAEvolutionStrategy(
        x0=[16, 48, 36],
        sigma0=5,
        inopts={
            "bounds":  [[0, 0, 0], [100, 100, 100]],
            "tolx":    0.1,
            "tolfun":  10,
            "maxiter": 200,
            "verbose": -9
        }
    )
    es.optimize(obj)
    r_opt, sc_opt, sp_opt = es.result.xbest

    # enforce sum <= 100
    total = r_opt + sc_opt + sp_opt
    if total > 100:
        r_opt  *= 100 / total
        sc_opt *= 100 / total
        sp_opt *= 100 / total

    return r_opt, sc_opt, sp_opt

# ── main pipeline ──────────────────────────────────────────────────────────────

# your three scenarios
scenarios = {
    # "uniform belief":        [1,  1,  1],
    # "naive dominated":       [20, 5,  1],
     "nash dominated":        [20, 5, 75],
    "nash dominated_2":      [10, 10, 80],
    "nash dominated_3":      [5, 5, 90],
}

# Nash SP* per scenario: CMA-ES result + 4pp stability correction
nash_sp_per_scenario = {
    "uniform belief":   None,          # did not converge — use 42% as midpoint estimate
    "naive dominated":  43.23 + 4,     # → 47%
    "nash dominated":   45.92 + 4,     # → 50%
    "nash dominated_2": 39.65 + 4,     # → 44%
    "nash dominated_3": 46.39 + 4,     # → 50%
}


results = {}

for name, alpha in scenarios.items():
    rational_sp = nash_sp_per_scenario[name]

    print(f"\n{'='*60}")
    print(f"Scenario: {name}   alpha={alpha}   Type C Nash SP={rational_sp}%")
    print(f"{'='*60}")

    # step 1: optimize with CMA-ES
    print("  Optimizing...")
    r_opt, sc_opt, sp_opt = optimize_cma(alpha, rational_sp, n_trials=2_000)

    # step 2: re-evaluate at optimum with more trials for clean distribution
    print("  Evaluating distribution at optimum...")
    result = monte_carlo_expected_pnl(
        r_opt, sc_opt, sp_opt,
        rational_sp=rational_sp,
        dirichlet_alpha=alpha,
        n_trials=10_000           # more trials for clean P10/P90
    )

    results[name] = {
        "r": r_opt, "sc": sc_opt, "sp": sp_opt,
        **{k: v for k, v in result.items() if k != "pnls"}
    }

    print(f"\n  Optimal allocation:")
    print(f"    R*     = {r_opt:.1f}%")
    print(f"    SC*    = {sc_opt:.1f}%")
    print(f"    SP*    = {sp_opt:.1f}%")
    print(f"    Total  = {r_opt+sc_opt+sp_opt:.1f}%")
    print(f"\n  PnL distribution:")
    print(f"    E[PnL] = {result['mean']:>12,.0f}")
    print(f"    Std    = {result['std']:>12,.0f}")
    print(f"    P10    = {result['p10']:>12,.0f}   ← bad day")
    print(f"    P90    = {result['p90']:>12,.0f}   ← good day")

df_results = pd.DataFrame(results).T
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(df_results[["r","sc","sp","mean","std","p10","p90"]].to_string())


Scenario: nash dominated   alpha=[20, 5, 75]   Type C Nash SP=49.92%
  Optimizing...
  Evaluating distribution at optimum...

  Optimal allocation:
    R*     = 15.2%
    SC*    = 43.3%
    SP*    = 41.5%
    Total  = 100.0%

  PnL distribution:
    E[PnL] =       46,742
    Std    =       10,831
    P10    =       33,065   ← bad day
    P90    =       60,903   ← good day

Scenario: nash dominated_2   alpha=[10, 10, 80]   Type C Nash SP=43.65%
  Optimizing...
  Evaluating distribution at optimum...

  Optimal allocation:
    R*     = 14.8%
    SC*    = 37.6%
    SP*    = 47.6%
    Total  = 100.0%

  PnL distribution:
    E[PnL] =      215,128
    Std    =        3,771
    P10    =      210,133   ← bad day
    P90    =      219,698   ← good day

Scenario: nash dominated_3   alpha=[5, 5, 90]   Type C Nash SP=50.39%
  Optimizing...
  Evaluating distribution at optimum...

  Optimal allocation:
    R*     = 16.1%
    SC*    = 44.2%
    SP*    = 39.7%
    Total  = 100.0%

  PnL distributio

### Visualization

In [61]:
def optimize_at_fixed_proportions(proportions, n_trials=500):
    """optimize given one specific proportion draw — no Dirichlet uncertainty"""
    def obj(x):
        pnls = [
            PnL_global(x[0], x[1], x[2], proportions)
            for _ in range(n_trials)
        ]
        return -np.mean(pnls)

    res = minimize(
        obj, x0=[16, 48, 36],
        method="SLSQP",
        bounds=[(0,100),(0,100),(0,100)],
        constraints=[{"type":"ineq","fun": lambda x: 100 - sum(x)}],
        options={"ftol":1e-4, "maxiter":200}
    )
    return res.x  # (R*, SC*, SP*)

# for one scenario, draw N proportion samples and optimize at each
alpha = [20, 5, 75]  # nash dominated
N_proportion_draws = 100

records = []
for _ in range(N_proportion_draws):
    props = np.random.dirichlet(alpha)
    r, sc, sp = optimize_at_fixed_proportions(props)
    records.append({
        "pi_a": props[0], "pi_b": props[1], "pi_c": props[2],
        "r": r, "sc": sc, "sp": sp
    })

df_stability = pd.DataFrame(records)

# violin plot of R*, SC*, SP* across proportion draws
fig = go.Figure()
for col, color, name in [("r","#1D9E75","Research %"),
                          ("sc","#378ADD","Scale %"),
                          ("sp","#E24B4A","Speed %")]:
    fig.add_trace(go.Violin(
        y=df_stability[col], name=name,
        box_visible=True, meanline_visible=True,
        fillcolor=color, opacity=0.6, line_color=color
    ))

fig.update_layout(
    title="Optimal allocation stability over proportion uncertainty (nash dominated)",
    yaxis_title="Optimal allocation %",
    height=450
)
fig.show()

In [62]:
records_all = []
for name, alpha in scenarios.items():
    for _ in range(N_proportion_draws):
        props = np.random.dirichlet(alpha)
        r, sc, sp = optimize_at_fixed_proportions(props)
        records_all.append({
            "scenario": name,
            "pi_a": props[0], "pi_b": props[1], "pi_c": props[2],
            "r": r, "sc": sc, "sp": sp
        })

df_all = pd.DataFrame(records_all)

# one subplot per allocation variable, one trace per scenario
fig = make_subplots(rows=1, cols=3,
    subplot_titles=["Research %", "Scale %", "Speed %"])

colors = {"uniform belief": "#888", 
          "naive dominated": "#1D9E75",
          "nash dominated": "#E24B4A"}

for col, plot_col in [("r",1), ("sc",2), ("sp",3)]:
    for scenario_name, group in df_all.groupby("scenario"):
        fig.add_trace(go.Violin(
            y=group[col],
            name=scenario_name,
            box_visible=True,
            meanline_visible=True,
            fillcolor=colors[scenario_name],
            opacity=0.6,
            line_color=colors[scenario_name],
            legendgroup=scenario_name,
            showlegend=(col == "r")   # only show legend once
        ), row=1, col=plot_col)

fig.update_layout(
    title="Optimal allocation stability across scenarios and proportion uncertainty",
    height=500
)
fig.show()